# Tutorial - Coliop - Parte 2c - Criando instâncias com **pyCmpl** no Google Colab

# 1. Configuração do Ambiente

Nesta etapa, configuramos o ambiente do Google Colab para utilizar a linguagem de otimização **CMPL**.

Essa célula deve ser executada apenas:
- Ao abrir o notebook pela primeira vez
- Ou quando o ambiente for reiniciado/desconectado

O que será feito:
- Download do COLIOP/CMPL (disponível em [coliop.org](https://www.coliop.org/download.html))
- Extração dos arquivos
- Configuração de variáveis de ambiente
- Inclusão das bibliotecas no Python

In [1]:
### Esta célula dever ser executada apenas quando o notebook for aberto
### ou quando o ambiente for desconectado e for necessário reconectá-lo
### por inatividade!

# Importação de bibliotecas do sistema
import os, sys

# Baixa o pacote do CMPL a partir do site oficial
os.system("wget https://www.coliop.org/_download/Cmpl-2-1-0-linux64.tar.gz")

# Extrai o arquivo baixado
os.system("tar -xzf Cmpl-2-1-0-linux64.tar.gz")

# Define o diretório principal do CMPL
os.environ["CMPLHOME"] = "/content/Cmpl-2-1-0-linux64"

# Adiciona o executável do CMPL ao PATH do sistema
os.environ["PATH"] += ":/content/Cmpl-2-1-0-linux64/bin"

# Configura bibliotecas compartilhadas necessárias
os.environ["LD_LIBRARY_PATH"] = "/content/Cmpl-2-1-0-linux64/bin/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Adiciona os módulos Python do CMPL ao caminho de importação
sys.path.append("/content/Cmpl-2-1-0-linux64/pyCmpl")
sys.path.append("/content/Cmpl-2-1-0-linux64/pyCmpl/lib3")
sys.path.append("/content/Cmpl-2-1-0-linux64/pyCmpl/scripts/Unix")

# 2. Definição do Modelo de Otimização (Exemplo 1 da Aula 03 do Tutorial)

Aqui definimos um problema de **programação linear inteira** usando a linguagem do CMPL.

## 2.1 Objetivo:
Minimizar o custo total de produtos.

## 2.2 Elementos do modelo:
- **Plantas**: Conjunto de pontos de oferta (plantas).
- **Centros**: Conjunto de pontos de demanda (centros).
- **Custos**: custos por unidade enviada.
- **Oferta**: Quantidade ofertada pelas plantas de produção.
- **Demanda**: Quantidade demandada pelos centros de consumo.

## 2.3 Variável de decisão:
- `x[j]`: quantidade enviadas da planta `𝑖` para o centro `𝑗`.

In [2]:
# Definição do modelo geral (generalizado) em linguagem CMPL (como string)
codigo_cmpl = """
%solver highs  // Resolvedor adotado: HiGHS - High Performance Optimization Software
%opt highs solver='choose' ### Opções do resolvedor adotado. Parâmetro interno solver: "choose", "simplex", "ipm", "ipx", "hipo" or "pdlp". ### solver='choose' define internamente se o problema é um LP, MIP ou QP.
par:
// Conjuntos
I:=set(1..3); // conjunto de plantas
J:=set(1..4); // conjunto de centros

// Parametros
O[I]:=(5000,6000,2500); // Oferta em unidades da planta i
D[J]:=(6000,4000,2000,1500); // Demanda em unidades do centro j
C[I,J]:=((3,2,7,6),(7,5,2,3),(2,5,4,5)); // Custo em unidades de envio da planta i para o centro j

var:
x[I,J]: int[0..]; // Numero de unidades enviadas da planta 𝑖 para o centro 𝑗

obj:FO: sum{i in I, j in J: C[i,j] * x[i,j] } ->min; // minimizar o custo total

con:
Ofertas {i in I : sum{j in J: x[i,j]} = O[i];} //Restricoes de Oferta
Demandas {j in J: sum{i in I: x[i,j]} = D[j]; } //Restricoes de Demanda

"""

# Salva o modelo em um arquivo .cmpl
with open("/content/problema_de_transporte_unico.cmpl", "w") as f:
    f.write(codigo_cmpl)

# 3. Resolução do Modelo

Nesta etapa utilizamos a **API Python pyCmpl** para fazer a integração entre modelo e resolvedor:
1. Carregamos o modelo criado
2. Executamos o solver
3. Exibimos o relatório da solução
4. Tratamento de erro para capturar possíveis problemas.

Também é possível fazer a entrada de dados por um arquivo python, atribuindo valores para conjuntos e parâmetros do modelo, utilizando atributos e métodos do pyCmpl. Isto será objeto para outra aula.

In [4]:
# Importa a biblioteca do CMPL para Python
from pyCmpl import *

try:
    # Cria o objeto do modelo a partir do arquivo
    modelo = Cmpl("/content/problema_de_transporte_unico.cmpl")

    # Resolve o problema de otimização
    modelo.solve()

    # Exibe o relatório da solução (valores ótimos)
    modelo.solutionReport()

except CmplException as e:
    # Caso ocorra erro, exibe a mensagem
    print(e.msg)

---------------------------------------------------------------------------------------------------------
Problem              problema_de_transporte_unico.cmpl
Nr. of variables     12
Nr. of constraints   7
Objective name       FO
Solver name          HIGHS
Display variables    (all)
Display constraints  (all)
---------------------------------------------------------------------------------------------------------

Objective status     INTEGER OPTIMAL
Objective value      39500.00            (min!)

Variables           
Name                 Type            Activity          LowerBound          UpperBound            Marginal
---------------------------------------------------------------------------------------------------------
x[1,1]                  I                3500                0.00                 inf                   -
x[1,2]                  I                1500                0.00                 inf                   -
x[1,3]                  I                   0    

# Source code

The source code is available on Github:
Cmpl official realeases: https://github.com/coin-or/Cmpl

Cmpl working git: https://github.com/MikeSteglich/Cmpl2

Coliop: https://github.com/MikeSteglich/Coliop

pyCmpl: https://github.com/MikeSteglich/pyCmpl3